In [10]:
import os
import time
import io
import pandas as pd
from PIL import Image
from google import genai
from google.genai import types
from dotenv import load_dotenv

In [14]:
# .env 파일의 환경변수를 시스템 환경변수로 로드
load_dotenv()

# os.environ.get()을 통해 안전하게 가져오기
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")

# # 코드 상에는 실제 키가 노출되지 않습니다.
# 확인용 출력
# print(f"불러온 API Key: {OPENAI_API_KEY}") 

In [ ]:
DATASET_PATH = "/home/minkyujeong/.cache/kagglehub/datasets/raddar/chest-xrays-indiana-university/versions/2" 

# =====================================================================
# [설정 영역] 본인의 환경에 맞게 경로와 API 키를 입력하세요.
# =====================================================================

CSV_PATH = "/home/minkyujeong/work/mini_thon/VLM/data/final_chest_xray_test_800_unique.csv"       # 원본 800장 CSV 파일 경로
IMAGE_DIR = "/home/minkyujeong/.cache/kagglehub/datasets/raddar/chest-xrays-indiana-university/versions/2/images/images_normalized"             # .png 이미지들이 들어있는 폴더 경로
OUTPUT_CSV_PATH = "vlm_results.csv" # 최종 결과가 저장될 CSV 파일 이름
# =====================================================================

In [ ]:
# 1. 최신 문법으로 제미나이 클라이언트 및 설정 선언
client = genai.Client(api_key=OPENAI_API_KEY)

# 정량적 비교 실험을 위해 무작위성(temperature)을 0으로 고정합니다.
config = types.GenerateContentConfig(temperature=0.0)
# MODEL_NAME = 'gemini-3.5-flash'
MODEL_NAME = 'gemini-3.1-flash-lite'

In [ ]:
# 프롬프트 v1
# SYSTEM_PROMPT = """[Role]
# You are an expert thoracic radiologist. Analyze the provided Frontal View Chest X-ray image with high clinical accuracy and generate a structured radiological report.

# [Context]
# - Image Type: Chest X-ray
# - Projection: Frontal View (PA/AP)
# - Anatomical Region: Thorax (Lungs, Heart, Mediastinum, Pleura, and Bones)

# [Task Instructions]
# 1. Observe the chest X-ray carefully and identify any abnormal findings, lesions, fractures, cardiomegaly, or pathological signs.
# 2. If the lungs and surrounding thoracic structures appear entirely clear and healthy, explicitly state that the image appears normal.
# 3. Use standard, professional English medical terminology (e.g., opacity, infiltration, pleural effusion, nodule, etc.).
# 4. Do NOT make a definitive final diagnosis or overall clinical impression. Describe ONLY the physical, observational findings and patterns.

# [Output Format]
# - Write your findings strictly in ONE short plain text paragraphs. 
# - Keep the entire response under 2-3 sentences if possible.
# - Do NOT repeat the same sentence or phrase within the response.
# - Do NOT use any bullet points, numbering, or markdown formatting symbols (like -, *, or 1.).
# - Do NOT include any greetings, introductory remarks, or concluding text. Start directly with the radiological findings."""



In [ ]:
# 프롬프트 v2
SYSTEM_PROMPT = """[Role]
You are an expert thoracic radiologist. Analyze the provided Frontal View Chest X-ray image with high clinical accuracy and generate a structured radiological report.

[Context]
- Image Type: Chest X-ray
- Projection: Frontal View (PA/AP)
- Anatomical Region: Thorax (Lungs, Heart, Mediastinum, Pleura, and Bones)

[Task Instructions]
1. Observe the chest X-ray carefully and identify any abnormal findings, lesions, fractures, cardiomegaly, or pathological signs.
2. NEVER rely on a generic, pre-defined template statement. Every sentence must reflect the specific visual pixel nuances of this exact image.
3. Inspect the following regions with extreme micro-scrutiny before declaring it normal: the lung apices, peripheral lung fields, costophrenic angles, right and left hilar contours, and the entire visible thoracic spine track.
4. If the lungs and surrounding thoracic structures appear entirely clear and healthy, explicitly state that the image appears normal.
5. Use standard, professional English medical terminology (e.g., opacity, infiltration, pleural effusion, nodule, etc.).
6. Do NOT make a definitive final diagnosis. Instead, describe the observational findings and patterns.

[Output Format]
- Write your findings strictly in ONE short plain text paragraphs. 
- Keep the entire response under 2-3 sentences if possible.
- Do NOT repeat the same sentence or phrase within the response.
- Do NOT use any bullet points, numbering, or markdown formatting symbols (like -, *, or 1.).
- Do NOT include any greetings, introductory remarks, or concluding text. Start directly with the radiological findings."""

In [7]:
# 3. 최신 원본 800개 목록을 '무조건' 먼저 로드합니다.
print(f"📄 최신 원본 목록 {CSV_PATH} 파일을 로드합니다.")
df = pd.read_csv(CSV_PATH)

# 기존에 수집된 데이터가 있다면 원본 목록 뒤에 매핑해줍니다.
if os.path.exists(OUTPUT_CSV_PATH):
    print(f"🔄 기존 결과 파일({OUTPUT_CSV_PATH})을 발견했습니다. 수집된 데이터를 최신 목록에 매핑합니다.")
    df_old = pd.read_csv(OUTPUT_CSV_PATH)
    
    # 예전 파일에 vlm_output 컬럼이 존재하는 경우에만 가져옵니다.
    if 'vlm_output' in df_old.columns:
        # uid와 vlm_output만 쏙 골라내고, 빈 값은 제거합니다.
        old_results = df_old[['uid', 'vlm_output']].dropna(subset=['vlm_output'])
        old_results = old_results[old_results['vlm_output'].str.strip() != ""]
        
        # 최신 800개 원본 목록(df)을 기준으로, 기존 결과(old_results)를 uid 매칭으로 합칩니다.
        df = pd.merge(df, old_results, on='uid', how='left')
    else:
        df['vlm_output'] = None
else:
    # 최초 실행인 경우 결과 컬럼을 새로 생성합니다.
    df['vlm_output'] = None

total_rows = len(df)
print(f"📊 총 대상 데이터: {total_rows}건")

📄 최신 원본 목록 /home/minkyujeong/work/mini_thon/VLM/data/final_chest_xray_test_800_unique.csv 파일을 로드합니다.
📊 총 대상 데이터: 800건


In [8]:
df.head()

,uid,filename,projection,MeSH,Problems,image,indication,comparison,findings,impression,rough_label,label,vlm_output
0,3252,3252_IM-1542-1001.dcm.png,Frontal,normal,normal,Xray Chest PA and Lateral,ECF placement.,Chest x-XXXX of XXXX,No evidence of airspace opacity. No effusion o...,No acute cardiopulmonary abnormality. .,Abnormal,Normal,None
1,2039,2039_IM-0683-1001.dcm.png,Frontal,Calcified Granuloma/lung/hilum/right/multiple/...,Calcified Granuloma;Spinal Fusion,Xray Chest PA and Lateral,Shortness of breath after cervical spine surgery.,None.,Cardiac and mediastinal contours are within no...,No acute pulmonary findings.,Normal,Abnormal,None
2,1594,1594_IM-0385-1001.dcm.png,Frontal,normal,normal,PA and lateral chest radiographs.,XXXX-year-old male with chest pain.,None.,The heart and cardiomediastinal silhouette are...,No acute cardiopulmonary finding.,Abnormal,Normal,None
3,718,718_IM-2280-1001.dcm.png,Frontal,Scoliosis/mild,Scoliosis,Xray Chest PA and Lateral,XXXX for 3 weeks,NaN,The lungs are clear. There is no pleural effus...,No acute pulmonary disease.,Abnormal,Abnormal,None
4,3593,3593_IM-1772-1001.dcm.png,Frontal,normal,normal,"and lateral chest XXXX, XXXX at XXXX for comp...",hemoptysis.,NaN,Heart size normal. Lungs are clear. XXXX are n...,Normal chest,Abnormal,Normal,None


In [9]:
# 4. 루프 가동
for idx, row in df.iterrows():
    # ⚠️ 테스트 세션: 최초 3개 검증 완료 후 전체 800장을 돌릴 때는 아래 두 줄을 지우거나 주석 처리(#)하세요!
    # if idx >= 3: 
    #     break
        
    uid = row['uid']
    filename = row['filename']
    img_path = os.path.join(IMAGE_DIR, filename)
    
    # 실시간 진행 상태 계산 및 출력 준비
    progress_percent = ((idx + 1) / total_rows) * 100
    print(f"⏳ [{idx+1}/{total_rows}] ({progress_percent:.1f}%) | 현재 작업 중: {filename} (UID: {uid}) ...", end="\r", flush=True)
        
    # 🌟 [이어받기 검증 수선] 결과가 존재하고, 'ERROR:'로 시작하지 않는 정상 판독문인 경우에만 패스합니다.
    has_output = pd.notna(row['vlm_output']) and str(row['vlm_output']).strip() != ""
    is_error = has_output and str(row['vlm_output']).startswith("ERROR:")
    
    if has_output and not is_error:
        print(f"⏭️ [{idx+1}/{total_rows}] ({progress_percent:.1f}%) | 패스: {filename}는 이미 정상 완료된 데이터입니다.        ")
        continue
        
    # 만약 기존 세션에서 에러가 났던 케이스라면 화면에 알리고 재시도 프로세스 진입
    if is_error:
        print(f"🔄 [{idx+1}/{total_rows}] ({progress_percent:.1f}%) | 재시도: 과거 에러가 발생했던 {filename}를 다시 시도합니다.        ")
        
    if not os.path.exists(img_path):
        print(f"❌ [{idx+1}/{total_rows}] ({progress_percent:.1f}%) | 에러: 이미지를 찾을 수 없음: {filename}        ")
        df.at[idx, 'vlm_output'] = "ERROR: Image file not found"
        df.to_csv(OUTPUT_CSV_PATH, index=False, encoding='utf-8-sig')
        continue
        
    # 🌟 구글 서버 과부하(503/429) 대응용 자체 재시도 루프 구성
    max_retries = 3   # 서버 불안정 시 제자리에서 최대 3번 찔러봄
    retry_delay = 15  # 재시도 전 숨 고르기 대기 시간 (초)
    
    for attempt in range(max_retries):
        try:
            # 1. 이미지 로드 및 메모리 상 리사이징 (1024px 축소로 전송 병목 타파)
            img = Image.open(img_path)
            MAX_SIZE = 1024
            img.thumbnail((MAX_SIZE, MAX_SIZE), Image.Resampling.LANCZOS)
            
            # 2. 고속 바이너리 바이트(Bytes) 변환 파이프라인
            img_byte_arr = io.BytesIO()
            img_format = img.format if img.format else "JPEG"
            img.save(img_byte_arr, format=img_format)
            img_bytes = img_byte_arr.getvalue()
            
            image_part = types.Part.from_bytes(
                data=img_bytes,
                mime_type=f"image/{img_format.lower()}"
            )
            
            # 3. 최신 SDK 문법으로 제미나이 3.5 플래시 모델 호출
            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=[image_part, SYSTEM_PROMPT],
                config=config
            )
            vlm_text = response.text.strip()
            
            # 4. 결과 프레임 기록 및 유실 방지 즉시 세이브
            df.at[idx, 'vlm_output'] = vlm_text
            print(f"✅ [{idx+1}/{total_rows}] ({progress_percent:.1f}%) | 완료: {filename}                        ")
            df.to_csv(OUTPUT_CSV_PATH, index=False, encoding='utf-8-sig')
            
            time.sleep(1.5)  # 결제 계정(Paid Tier) 최적 효율 마진 대기
            break  # 호출에 성공했으므로 재시도 루프를 빠져나와 다음 이미지로 이동!
            
        except Exception as e:
            error_msg = str(e)
            
            # 🌟 [수정] NameError 방지를 위해 문자열 기반 및 속성 기반 검사로 완벽하게 우회합니다.
            is_server_busy = False
            
            # 1. 에러 메시지 텍스트에 과부하 관련 키워드가 있는지 검사
            if "503" in error_msg or "UNAVAILABLE" in error_msg or "429" in error_msg or "ResourceExhausted" in error_msg:
                is_server_busy = True
            # 2. 혹은 에러 객체 자체에 code 속성이 존재하고 그 값이 429나 503인 경우
            elif hasattr(e, 'code') and (getattr(e, 'code') in [429, 503]):
                is_server_busy = True
                
            if is_server_busy:
                if attempt < max_retries - 1:
                    print(f"\n⚠️ 구글 서버 과부하(503/429) 포착! {retry_delay}초 후 제자리에서 다시 찌릅니다... (재시도 {attempt+1}/{max_retries})")
                    time.sleep(retry_delay)
                    continue  # 다음 attempt(재시도) 루프 실행
                else:
                    # 3번 다 실패하면 에러 기록하고 패스 (추후 다시 켜면 이 에러 기록을 보고 재시도함)
                    print(f"\n❌ [{idx+1}/{total_rows}] 3회 재시도 모두 실패하여 다음으로 넘어갑니다: {filename}")
                    df.at[idx, 'vlm_output'] = f"ERROR: SERVER_DOWN_503_{error_msg}"
                    df.to_csv(OUTPUT_CSV_PATH, index=False, encoding='utf-8-sig')
            else:
                # 503/429 이외의 특수 에러(예: 로컬 파일 손상 등)는 즉시 에러 기록 후 통과
                print(f"❌ [{idx+1}/{total_rows}] 에러 발생: {filename} -> {error_msg}        ")
                df.at[idx, 'vlm_output'] = f"ERROR: {error_msg}"
                df.to_csv(OUTPUT_CSV_PATH, index=False, encoding='utf-8-sig')
                time.sleep(3)
                break  # attempt 루프를 깨고 완전히 다음 파일로 이동

print(f"\n🎉 스크립트 실행이 완전히 종료되었습니다. 최종 결과 파일 확인: {OUTPUT_CSV_PATH}")

✅ [1/800] (0.1%) | 완료: 3252_IM-1542-1001.dcm.png                        
✅ [2/800] (0.2%) | 완료: 2039_IM-0683-1001.dcm.png                        
✅ [3/800] (0.4%) | 완료: 1594_IM-0385-1001.dcm.png                        
✅ [4/800] (0.5%) | 완료: 718_IM-2280-1001.dcm.png                        
✅ [5/800] (0.6%) | 완료: 3593_IM-1772-1001.dcm.png                        
✅ [6/800] (0.8%) | 완료: 3294_IM-1573-1001.dcm.png                        
✅ [7/800] (0.9%) | 완료: 692_IM-2258-1001.dcm.png                        
✅ [8/800] (1.0%) | 완료: 983_IM-2471-1001.dcm.png                        
✅ [9/800] (1.1%) | 완료: 2644_IM-1130-1001.dcm.png                        
✅ [10/800] (1.2%) | 완료: 604_IM-2193-1001.dcm.png                        
✅ [11/800] (1.4%) | 완료: 731_IM-2291-1001.dcm.png                        
✅ [12/800] (1.5%) | 완료: 1190_IM-0128-1001.dcm.png                        
✅ [13/800] (1.6%) | 완료: 2562_IM-1065-1001.dcm.png                        
✅ [14/800] (1.8%) | 완료: 733_IM-2293-0001-0002.dcm.pn